# Entrenar YOLO con tu propio dataset

**Visión Artificial — CUTLAJO**

Un solo notebook que funciona igual en **Google Colab**, en **tu computadora** y en la
**DGX Spark** de la universidad. Detecta solo dónde está corriendo y ajusta lo necesario:

| | Colab | Tu computadora | DGX Spark |
|---|---|---|---|
| GPU | T4 (hay que prenderla) | la tuya, si tienes NVIDIA o Mac M | GB10, compartida |
| Dónde quedan los resultados | tu Google Drive | junto a este notebook | carpeta común del grupo |
| Batch automático | según la memoria de la T4 | según tu GPU | conservador (memoria unificada) |
| Turnos | no hacen falta | no hacen falta | uno entrena a la vez |
| Si se cae la sesión | checkpoint cada 5 épocas y reanudas | — | — |

### Cómo se usa

1. Corre las celdas **en orden**, de arriba a abajo.
2. **Solo editas el Paso 1** (configuración). Todo lo demás se ajusta solo.
3. Para comparar, regresa al Paso 1, cambia el modelo o el tamaño, y vuelve a correr
   del Paso 5 en adelante. Cada entrenamiento queda guardado como una *corrida* distinta.

¿No tienes dataset todavía? Deja `DATASET = "demo"` y el notebook genera uno de figuras
(círculos, cuadrados y triángulos) para que veas el flujo completo en unos minutos.

---
## Paso 0 — Instalar

En **Colab**, antes que nada: *Entorno de ejecución → Cambiar tipo de entorno de
ejecución → **T4 GPU***. Si no, entrena en CPU y tarda horas.

En **tu computadora**, desde la raíz del repo: `uv sync --group yolo` (una vez) y luego
`uv run jupyter lab`. Esta celda revisa que no falte nada e instala lo que haga falta.

In [ ]:
import importlib.util, os, platform, shutil, subprocess, sys

EN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
faltan = [m for m in ("ultralytics", "yaml", "pandas", "matplotlib")
          if importlib.util.find_spec(m) is None]

if faltan:
    print("Instalando lo que falta:", ", ".join(faltan), "...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "ultralytics", "pandas", "matplotlib", "pyyaml"])
    except Exception as e:
        print(f"No pude instalar con pip ({e}).")
        print("Si usas uv: cierra Jupyter, corre  uv sync --group yolo  en la raíz del repo y vuelve a abrir.")

import torch
print(f"PyTorch {torch.__version__}  ·  GPU NVIDIA: {torch.cuda.is_available()}")

# DGX Spark (ARM64): el PyTorch que trae pip no siempre ve la GPU.
if platform.machine() == "aarch64" and shutil.which("nvidia-smi") and not torch.cuda.is_available():
    print("\nHay GPU NVIDIA pero PyTorch no la ve. En la DGX Spark instala el de CUDA 13:")
    print("  pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130")
    print("y reinicia el kernel (Kernel → Restart).")

---
## Paso 1 — Configuración  ✏️

Es la **única celda que editas**. En Colab aparece como formulario con menús.

| Variable | Qué es |
|---|---|
| `ALUMNO` | Tu nombre o el de tu equipo. Sale en la tabla comparativa del grupo. |
| `DATASET` | `"demo"`, un `.zip` (Roboflow, SAM 3…), una carpeta o un `data.yaml`. Ver Paso 4. |
| `CLASES` | Solo si tu dataset **no** trae `data.yaml`: nombres separados por comas, en orden (`"humo,fuego"`). |
| `VERSION` / `TAMANO` | Qué YOLO y de qué tamaño. La tabla del Paso 5 dice qué combinaciones existen. |
| `TAREA` | `detect` (cajas) o `segment` (contornos). Las etiquetas de SAM 3 sirven para las dos. |
| `EPOCAS` | Pasadas completas por el dataset. 50 es un buen inicio; en CPU usa 5. |
| `IMGSZ` | Tamaño al que se reescalan las imágenes. Más grande = ve objetos chicos, pero más lento. |
| `BATCH` | Imágenes por paso. **0 = automático** según tu memoria. |
| `DISPOSITIVO` | `auto` casi siempre. `"0"`, `"1"` o `"0,1"` para elegir GPU; `mps` en Mac; `cpu` para forzar CPU. |

In [ ]:
#@title Configuración
ALUMNO      = "equipo1"   #@param {type:"string"}
DATASET     = "demo"      #@param {type:"string"}
CLASES      = ""          #@param {type:"string"}
VERSION     = "yolo11"    #@param ["yolov8", "yolov9", "yolov10", "yolo11", "yolo12", "yolo26"]
TAMANO      = "n"         #@param ["n", "s", "m", "l", "x", "t", "c", "e", "b"]
TAREA       = "detect"    #@param ["detect", "segment"]
EPOCAS      = 50          #@param {type:"integer"}
IMGSZ       = 640         #@param [320, 416, 512, 640, 800, 1024] {type:"raw"}
BATCH       = 0           #@param {type:"integer"}
DISPOSITIVO = "auto"      #@param ["auto", "0", "1", "0,1", "mps", "cpu"]

---
## Paso 2 — Funciones de apoyo

Córrela y sigue. Tiene todo lo que pasa "por debajo": detectar el equipo, acomodar el
dataset, entrenar, reanudar y comparar. Está oculta para que no estorbe; si te da
curiosidad, ábrela, no hay magia.

In [ ]:
#@title Funciones de apoyo  (corre esta celda y sigue; no hace falta leerla)
# ============================================================================
# Todo lo que hace el notebook "por debajo" vive en esta celda, para que el
# resto quede legible. Si te da curiosidad, léela: no hay magia, solo
# archivos, rutas y llamadas a Ultralytics.
# ============================================================================
from __future__ import annotations

import json
import os
import platform
import random
import re
import shutil
import sys
import time
import zipfile
from contextlib import contextmanager
from pathlib import Path

import yaml

# --------------------------------------------------------------------------- #
# Catálogo de modelos
# --------------------------------------------------------------------------- #
# Solo detección y segmentación: son las tareas que salen de SAM 3 y de
# Roboflow. Los tamaños van de menor a mayor.
CATALOGO = {
    "yolov8":  {"año": 2023, "sizes": ["n", "s", "m", "l", "x"],
                "tareas": ["detect", "segment"],
                "nota": "El clásico. Punto de comparación obligado."},
    "yolov9":  {"año": 2024, "sizes": ["t", "s", "m", "c", "e"],
                "tareas": ["detect", "segment"], "seg_sizes": ["c", "e"],
                "nota": "PGI y GELAN. En segmentación solo hay c y e."},
    "yolov10": {"año": 2024, "sizes": ["n", "s", "m", "b", "l", "x"],
                "tareas": ["detect"],
                "nota": "Sin NMS: menor latencia. Solo detección."},
    "yolo11":  {"año": 2024, "sizes": ["n", "s", "m", "l", "x"],
                "tareas": ["detect", "segment"],
                "nota": "El default razonable si no tienes razón para otro."},
    "yolo12":  {"año": 2025, "sizes": ["n", "s", "m", "l", "x"],
                "tareas": ["detect"],
                "nota": "Basado en atención. Solo detección."},
    "yolo26":  {"año": 2026, "sizes": ["n", "s", "m", "l", "x"],
                "tareas": ["detect", "segment"],
                "nota": "El más reciente. Pide ultralytics al día."},
}

# Parámetros aproximados (millones) del checkpoint de detección en COCO.
PARAMS_M = {
    "yolov8":  {"n": 3.2, "s": 11.2, "m": 25.9, "l": 43.7, "x": 68.2},
    "yolov9":  {"t": 2.0, "s": 7.2, "m": 20.1, "c": 25.5, "e": 58.1},
    "yolov10": {"n": 2.3, "s": 7.2, "m": 15.4, "b": 19.1, "l": 24.4, "x": 29.5},
    "yolo11":  {"n": 2.6, "s": 9.4, "m": 20.1, "l": 25.3, "x": 56.9},
    "yolo12":  {"n": 2.6, "s": 9.3, "m": 20.2, "l": 26.4, "x": 59.1},
    "yolo26":  {"n": 2.4, "s": 9.5, "m": 20.4, "l": 24.8, "x": 55.7},
}


def sizes_de(version: str, tarea: str) -> list[str]:
    info = CATALOGO[version]
    return info.get("seg_sizes", info["sizes"]) if tarea == "segment" else info["sizes"]


def nombre_pesos(version: str, size: str, tarea: str) -> str:
    """yolo11 + m + segment  ->  'yolo11m-seg.pt'"""
    if version not in CATALOGO:
        raise ValueError(f"No conozco '{version}'. Opciones: {', '.join(CATALOGO)}")
    if tarea not in CATALOGO[version]["tareas"]:
        raise ValueError(f"{version} no tiene pesos para '{tarea}'. "
                         f"Tiene: {', '.join(CATALOGO[version]['tareas'])}.")
    if size not in sizes_de(version, tarea):
        raise ValueError(f"{version} en '{tarea}' no tiene tamaño '{size}'. "
                         f"Tiene: {', '.join(sizes_de(version, tarea))}.")
    return f"{version}{size}{'-seg' if tarea == 'segment' else ''}.pt"


def mostrar_catalogo():
    import pandas as pd
    filas = [{"version": v, "año": i["año"],
              "tamaños detect": " ".join(i["sizes"]),
              "tamaños segment": " ".join(sizes_de(v, "segment")) if "segment" in i["tareas"] else "—",
              "nota": i["nota"]} for v, i in CATALOGO.items()]
    return pd.DataFrame(filas).set_index("version")


# --------------------------------------------------------------------------- #
# Entorno: dónde estamos corriendo y qué conviene
# --------------------------------------------------------------------------- #
def _meminfo_gb(clave: str):
    try:
        with open("/proc/meminfo") as f:
            for ln in f:
                if ln.startswith(clave + ":"):
                    return int(ln.split()[1]) / 1024 ** 2
    except Exception:
        pass
    try:
        import psutil
        vm = psutil.virtual_memory()
        return (vm.total if clave == "MemTotal" else vm.available) / 1024 ** 3
    except Exception:
        return None


def en_colab() -> bool:
    return "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ


def _carpeta_del_notebook() -> Path:
    """La carpeta donde está el .ipynb (Jupyter arranca ahí)."""
    return Path.cwd().resolve()


def preparar_entorno(dispositivo: str = "auto", usar_drive: bool = True) -> dict:
    """
    Detecta el equipo y decide rutas, dispositivo y cuántos procesos de carga
    usar. Devuelve un diccionario que usa el resto del notebook.
    """
    E = {"tipo": "cpu", "avisos": [], "arquitectura": platform.machine(),
         "sistema": platform.system()}

    # ---- Rutas ---------------------------------------------------------- #
    if en_colab():
        E["tipo"] = "colab"
        raiz = Path("/content/entrenar_yolo")
        if usar_drive:
            try:
                from google.colab import drive
                if not Path("/content/drive/MyDrive").exists():
                    drive.mount("/content/drive")
                raiz = Path("/content/drive/MyDrive/VisionArtificial/entrenar_yolo")
            except Exception as e:
                E["avisos"].append(f"No pude montar Drive ({e}). Los resultados se "
                                   "BORRAN al cerrar Colab: descárgalos al terminar.")
    else:
        raiz = _carpeta_del_notebook()

    # Variables de entorno para la DGX: servir_dgx.sh las define para que
    # todo el grupo comparta corridas y pesos base.
    E["raiz"] = raiz
    E["datos"] = Path(os.environ.get("VA_DATOS", raiz / "datos"))
    E["runs"] = Path(os.environ.get("VA_RUNS", raiz / "runs"))
    E["pesos"] = Path(os.environ.get("VA_PESOS", raiz / "pesos"))
    for k in ("datos", "runs", "pesos"):
        E[k].mkdir(parents=True, exist_ok=True)

    # ---- GPU ------------------------------------------------------------ #
    import torch
    E["torch"] = torch.__version__
    try:
        import ultralytics
        E["ultralytics"] = ultralytics.__version__
    except ImportError:
        E["ultralytics"] = None
        E["avisos"].append("Falta ultralytics: corre la celda del Paso 0.")

    E["memoria_gb"], E["memoria_fuente"] = None, ""
    if torch.cuda.is_available():
        n = torch.cuda.device_count()
        props = [torch.cuda.get_device_properties(i) for i in range(n)]
        E["gpus"] = [(i, p.name, p.total_memory / 1024 ** 3) for i, p in enumerate(props)]
        cap = (props[0].major, props[0].minor)
        es_spark = "GB10" in props[0].name or (E["arquitectura"] == "aarch64" and cap == (12, 1))
        if E["tipo"] != "colab":
            E["tipo"] = "dgx_spark" if es_spark else "cuda"
        libre, _ = torch.cuda.mem_get_info(0)
        E["memoria_gb"], E["memoria_fuente"] = libre / 1024 ** 3, "memoria libre de la GPU 0"
        if es_spark:
            E["avisos"].append(
                "DGX Spark: la memoria es UNIFICADA (RAM y GPU son la misma). Un batch "
                "de más no da un error limpio, puede congelar la máquina de todo el "
                "grupo. Por eso el batch automático es conservador y se entrena por turnos.")
            E["avisos"].append(
                "Si PyTorch avisa que la capacidad 12.1 está fuera de rango, ignóralo: "
                "es un aviso conocido del GB10.")
        if n > 1:
            E["avisos"].append(
                "Hay más de una GPU. DISPOSITIVO='0,1' las usa juntas, pero la más "
                "lenta marca el ritmo. Con tarjetas distintas suele convenir lanzar "
                "dos corridas a la vez, una por tarjeta ('0' y '1').")
        disp_auto = "0"
    elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        E["tipo"] = "mps" if E["tipo"] != "colab" else "colab"
        E["memoria_gb"] = (_meminfo_gb("MemAvailable") or 8.0) * 0.6
        E["memoria_fuente"] = "memoria compartida de la Mac (60%)"
        E["avisos"].append("GPU de Apple (MPS): funciona, pero es lenta. Úsala para "
                           "probar el flujo con pocas épocas.")
        disp_auto = "mps"
    else:
        disp_auto = "cpu"
        if E["tipo"] == "colab":
            E["avisos"].append(
                "¡Colab está SIN GPU! Entorno de ejecución → Cambiar tipo de entorno "
                "de ejecución → T4 GPU, y vuelve a correr desde el Paso 0.")
        elif E["arquitectura"] == "aarch64" and shutil.which("nvidia-smi"):
            E["avisos"].append(
                "Hay GPU NVIDIA pero PyTorch no la ve: en la DGX Spark hay que instalar "
                "el PyTorch para CUDA 13 (ver Paso 0).")
        else:
            E["avisos"].append("Sin GPU: va a ser LENTO. Usa tamaño 'n', imgsz 320 "
                               "y pocas épocas, solo para ver que el flujo corre.")
        E["memoria_gb"] = _meminfo_gb("MemAvailable")
        E["memoria_fuente"] = "RAM libre"

    d = str(dispositivo).strip().lower()
    E["dispositivo"] = disp_auto if d in ("", "auto") else d
    # Si eligieron GPU(s) específicas, el batch se calcula con la de MENOS memoria libre
    ids = [x for x in E["dispositivo"].replace(" ", "").split(",") if x.isdigit()]
    if ids and torch.cuda.is_available():
        try:
            E["memoria_gb"] = min(torch.cuda.mem_get_info(int(i))[0] for i in ids) / 1024 ** 3
            E["memoria_fuente"] = f"memoria libre de la GPU {'/'.join(ids)}"
        except Exception as e:
            E["avisos"].append(f"No pude leer la GPU '{E['dispositivo']}': {e}. ¿Existe?")

    # ---- Procesos de carga de datos ------------------------------------- #
    cpus = os.cpu_count() or 2
    if E["tipo"] == "colab":
        E["workers"] = 2
    elif E["tipo"] in ("mps", "cpu"):
        E["workers"] = min(4, cpus)
    else:
        E["workers"] = min(8, cpus)

    # ---- Turnos: solo en la máquina compartida -------------------------- #
    E["turnos"] = E["tipo"] == "dgx_spark" or os.environ.get("VA_TURNOS") == "1"

    # ---- Reporte -------------------------------------------------------- #
    nombres = {"colab": "Google Colab", "dgx_spark": "NVIDIA DGX Spark",
               "cuda": "PC con GPU NVIDIA", "mps": "Mac con Apple Silicon", "cpu": "Solo CPU"}
    print("=" * 66)
    print(f"  Equipo        : {nombres[E['tipo']]}  ({E['sistema']}, {E['arquitectura']})")
    print(f"  PyTorch       : {E['torch']}    Ultralytics: {E['ultralytics']}")
    for i, nom, gb in E.get("gpus", []):
        print(f"  GPU {i}         : {nom}  ({gb:.0f} GB)")
    if E["memoria_gb"]:
        print(f"  Para entrenar : {E['memoria_gb']:.1f} GB  ({E['memoria_fuente']})")
    print(f"  Dispositivo   : {E['dispositivo']}    workers: {E['workers']}"
          f"    turnos: {'sí' if E['turnos'] else 'no'}")
    print(f"  Datos         : {E['datos']}")
    print(f"  Corridas      : {E['runs']}")
    print("=" * 66)
    for a in E["avisos"]:
        print(f"  [!] {a}")
    return E


def batch_automatico(E: dict, size: str, imgsz: int, n_train: int | None = None) -> int:
    """
    Batch conservador según la memoria disponible y el tamaño del modelo.
    Siempre se queda corto a propósito: en la Spark pasarse congela la máquina.
    """
    if E["dispositivo"] == "cpu":
        b = 8
    else:
        libre = E.get("memoria_gb") or 8.0
        margen = 0.25 if E["tipo"] == "dgx_spark" else 0.6
        costo = {"n": .09, "t": .09, "s": .16, "m": .30, "b": .36,
                 "c": .38, "l": .45, "e": .70, "x": .70}.get(size, .30)
        por_img = costo * (imgsz / 640) ** 2
        b = int(libre * margen / por_img)
    tope = 64
    if n_train:
        tope = min(tope, max(2, n_train // 2))
    for p in (64, 32, 16, 8, 4, 2):
        if b >= p and p <= tope:
            return p
    return 2


# --------------------------------------------------------------------------- #
# Dataset
# --------------------------------------------------------------------------- #
EXT_IMG = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def _limpiar(nombre: str) -> str:
    """'Ana López' -> 'Ana_Lopez': sin acentos ni espacios, seguro como nombre de carpeta."""
    import unicodedata
    nombre = unicodedata.normalize("NFKD", nombre).encode("ascii", "ignore").decode()
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", nombre).strip("_") or "dataset"


def _imagenes(carpeta: Path) -> list[Path]:
    return sorted(p for p in carpeta.iterdir() if p.suffix.lower() in EXT_IMG) \
        if carpeta.is_dir() else []


def _carpeta_labels(carpeta_img: Path) -> Path:
    """YOLO busca las etiquetas cambiando el último /images/ por /labels/."""
    partes = list(carpeta_img.parts)
    for i in range(len(partes) - 1, -1, -1):
        if partes[i] == "images":
            partes[i] = "labels"
            return Path(*partes)
    return carpeta_img.parent / "labels"


def _nombres_clases(cfg_names) -> list[str]:
    if isinstance(cfg_names, dict):
        return [str(cfg_names[k]) for k in sorted(cfg_names, key=lambda k: int(k))]
    return [str(c) for c in (cfg_names or [])]


def generar_demo(destino: Path, n: int = 160, lado: int = 320, semilla: int = 0) -> Path:
    """
    Dataset de juguete: círculos, cuadrados y triángulos sobre fondo con ruido.
    Sirve para probar todo el flujo en minutos, sin tener dataset propio.
    Las etiquetas son polígonos, así que sirve para detect y segment.
    """
    import numpy as np
    import cv2
    rng = np.random.default_rng(semilla)
    clases = ["circulo", "cuadrado", "triangulo"]
    for split, cuantas in (("train", int(n * 0.8)), ("val", n - int(n * 0.8))):
        (destino / "images" / split).mkdir(parents=True, exist_ok=True)
        (destino / "labels" / split).mkdir(parents=True, exist_ok=True)
        for k in range(cuantas):
            img = rng.integers(60, 140, (lado, lado, 3), dtype=np.uint8)
            img = cv2.GaussianBlur(img, (7, 7), 0)
            lineas = []
            for _ in range(rng.integers(1, 4)):
                c = int(rng.integers(0, 3))
                r = int(rng.integers(lado // 14, lado // 6))
                cx, cy = (int(v) for v in rng.integers(r + 2, lado - r - 2, 2))
                color = tuple(int(v) for v in rng.integers(150, 256, 3))
                mask = np.zeros((lado, lado), np.uint8)
                if c == 0:
                    cv2.circle(mask, (cx, cy), r, 255, -1)
                elif c == 1:
                    ang = float(rng.uniform(0, 90))
                    pts = cv2.boxPoints(((cx, cy), (2 * r, 2 * r), ang)).astype(np.int32)
                    cv2.fillPoly(mask, [pts], 255)
                else:
                    a0 = rng.uniform(0, 2 * np.pi)
                    pts = np.array([[cx + r * np.cos(a0 + j * 2 * np.pi / 3),
                                     cy + r * np.sin(a0 + j * 2 * np.pi / 3)]
                                    for j in range(3)], np.int32)
                    cv2.fillPoly(mask, [pts], 255)
                img[mask > 0] = color
                cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                if not cnts:
                    continue
                poli = max(cnts, key=cv2.contourArea).reshape(-1, 2) / lado
                lineas.append(f"{c} " + " ".join(f"{v:.5f}" for v in poli.flatten()))
            nombre = f"{split}_{k:04d}"
            cv2.imwrite(str(destino / "images" / split / f"{nombre}.jpg"), img)
            (destino / "labels" / split / f"{nombre}.txt").write_text("\n".join(lineas) + "\n")
    cfg = {"path": str(destino), "train": "images/train", "val": "images/val",
           "names": dict(enumerate(clases))}
    (destino / "data.yaml").write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))
    return destino / "data.yaml"


def _extraer_zip(zip_path: Path, datos: Path) -> Path:
    destino = datos / _limpiar(zip_path.stem)
    if not destino.exists():
        print(f"Descomprimiendo {zip_path.name} ...")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(destino)
    # Si el zip traía una sola carpeta adentro, entrar a ella
    hijos = [p for p in destino.iterdir() if not p.name.startswith((".", "__MACOSX"))]
    return hijos[0] if len(hijos) == 1 and hijos[0].is_dir() else destino


def _buscar_yaml(carpeta: Path) -> Path | None:
    candidatos = sorted(carpeta.rglob("*.yaml"), key=lambda p: (len(p.parts), p.name != "data.yaml"))
    candidatos = [p for p in candidatos if "names" in (yaml.safe_load(p.read_text(encoding="utf-8")) or {})]
    return candidatos[0] if candidatos else None


def _resolver_split(base: Path, carpeta_yaml: Path, valor) -> list[Path]:
    """
    Una entrada de train/val puede ser texto o lista, relativa o absoluta.
    Probamos las interpretaciones que de verdad aparecen:
      - relativa a 'path' del yaml           (la oficial)
      - relativa a la carpeta del yaml       (lo que la gente asume)
      - con '../' de más                     (exports de Roboflow)
    """
    if valor is None:
        return []
    salida = []
    for v in (valor if isinstance(valor, list) else [valor]):
        v = str(v)
        p = Path(v).expanduser()
        if p.is_absolute():
            salida.append(p)
            continue
        candidatos = [base / v, carpeta_yaml / v]
        if v.startswith("../"):
            candidatos += [base / v[3:], carpeta_yaml / v[3:]]
        existente = next((c.resolve() for c in candidatos if c.exists()), None)
        salida.append(existente or (base / v).resolve())
    return salida


def _dividir(imagenes: list[Path], destino: Path, clases: list[str],
             val: float = 0.2, semilla: int = 42) -> Path:
    """Reparte imágenes en train/val estratificando por la clase dominante."""
    def dominante(img):
        lbl = _carpeta_labels(img.parent) / f"{img.stem}.txt"
        cuenta = {}
        if lbl.exists():
            for ln in lbl.read_text(encoding="utf-8").splitlines():
                if ln.split():
                    c = ln.split()[0]
                    cuenta[c] = cuenta.get(c, 0) + 1
        return max(cuenta, key=cuenta.get) if cuenta else "_fondo"

    grupos = {}
    for img in imagenes:
        grupos.setdefault(dominante(img), []).append(img)
    rng = random.Random(semilla)
    reparto = {"train": [], "val": []}
    for _, g in sorted(grupos.items()):
        rng.shuffle(g)
        n_val = max(1, round(len(g) * val)) if len(g) > 1 else 0
        reparto["val"] += g[:n_val]
        reparto["train"] += g[n_val:]

    # Si ya se dividió antes con las mismas imágenes, se reutiliza (en Colab
    # reescribir todo en Drive cada sesión es lento). La semilla fija hace
    # que el reparto sea siempre el mismo.
    previas = {p.name for sp in ("train", "val") for p in _imagenes(destino / "images" / sp)}
    if previas and previas == {i.name for i in imagenes}:
        print(f"  Reutilizo la división que ya existía en {destino}")
        return destino
    if destino.exists():
        shutil.rmtree(destino)
    for split, lista in reparto.items():
        (destino / "images" / split).mkdir(parents=True)
        (destino / "labels" / split).mkdir(parents=True)
        for img in lista:
            shutil.copy2(img, destino / "images" / split / img.name)
            lbl = _carpeta_labels(img.parent) / f"{img.stem}.txt"
            if lbl.exists():
                shutil.copy2(lbl, destino / "labels" / split / lbl.name)
    print(f"  Dividí {len(imagenes)} imágenes: {len(reparto['train'])} train / "
          f"{len(reparto['val'])} val  ->  {destino}")
    return destino


def preparar_dataset(E: dict, entrada: str, clases_extra: str = "",
                     val: float = 0.2, copiar_local: bool = True) -> Path:
    """
    Recibe lo que tengas y devuelve un data.yaml listo para entrenar:
      'demo'                      -> genera un dataset de figuras
      algo.zip                    -> lo descomprime (export de Roboflow, de SAM 3...)
      una carpeta                 -> busca su data.yaml adentro
      un data.yaml                -> lo usa
    Arregla rutas relativas, divide train/val si hace falta (el export de
    SAM 3 trae train y val iguales) y escribe un yaml con rutas absolutas.
    """
    datos = E["datos"]
    entrada = str(entrada).strip().strip('"').strip("'")

    if entrada.lower() == "demo":
        destino = datos / "demo_figuras"
        if not (destino / "data.yaml").exists():
            print("Generando dataset de demostración (figuras)...")
            generar_demo(destino)
        ruta = destino / "data.yaml"
    else:
        ruta = Path(entrada).expanduser()
        if not ruta.is_absolute():
            for base in (Path.cwd(), E["raiz"], datos):
                if (base / ruta).exists():
                    ruta = base / ruta
                    break
        if not ruta.exists():
            raise FileNotFoundError(
                f"No encuentro '{entrada}'. Revisa la ruta. Busqué en la carpeta del "
                f"notebook, en {E['raiz']} y en {datos}.")
        ruta = ruta.resolve()
        if ruta.suffix.lower() == ".zip":
            ruta = _extraer_zip(ruta, datos)

    # ---- Encontrar el yaml (o fabricarlo) ------------------------------- #
    if ruta.is_dir():
        y = _buscar_yaml(ruta)
        if y is None:
            if not (ruta / "images").is_dir():
                raise FileNotFoundError(
                    f"En {ruta} no hay data.yaml ni carpeta images/. "
                    "Necesito al menos images/ y labels/.")
            clases = [c.strip() for c in clases_extra.split(",") if c.strip()]
            if not clases and (ruta / "classes.txt").exists():
                clases = [c.strip() for c in (ruta / "classes.txt").read_text().splitlines() if c.strip()]
            if not clases:
                raise ValueError("No hay data.yaml: escribe los nombres de las clases en "
                                 "CLASES, separados por comas, en el orden de sus índices.")
            # El yaml se escribe en datos/, no en tu carpeta: no tocamos tus archivos
            cfg = {"path": str(ruta), "train": "images", "val": "images", "names": dict(enumerate(clases))}
            (datos / "_yaml").mkdir(exist_ok=True)
            y = datos / "_yaml" / f"{_limpiar(ruta.name)}.origen.yaml"
            y.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))
            print(f"  No había data.yaml; uso las clases: {', '.join(clases)}")
        ruta = y

    cfg = yaml.safe_load(ruta.read_text(encoding="utf-8")) or {}
    clases = _nombres_clases(cfg.get("names"))
    if not clases:
        raise ValueError(f"{ruta} no declara clases en 'names'.")
    nombre = ruta.parent.name if ruta.name == "data.yaml" else ruta.stem.replace(".origen", "")
    if nombre.lower() in ("yolo", "dataset", "data", "datos", "export"):
        nombre = f"{ruta.parent.parent.name}_{nombre}"     # p. ej. salida/yolo de SAM 3
    nombre = _limpiar(nombre)
    base = Path(str(cfg.get("path") or "."))
    if not base.is_absolute():
        base = (ruta.parent / base).resolve()
    if not base.exists():
        base = ruta.parent.resolve()

    train = _resolver_split(base, ruta.parent, cfg.get("train"))
    valid = _resolver_split(base, ruta.parent, cfg.get("val") or cfg.get("valid"))
    faltan = [str(p) for p in train + valid if not p.exists()]
    if not train:
        raise ValueError(f"{ruta} no define 'train'.")
    if faltan:
        raise FileNotFoundError("Estas carpetas del data.yaml no existen:\n  " + "\n  ".join(faltan))

    # ---- ¿Hace falta dividir? ------------------------------------------- #
    mismo = not valid or {str(p) for p in train} == {str(p) for p in valid}
    if mismo:
        motivo = ("no hay 'val'" if not valid else
                  "la carpeta no viene dividida" if ruta.name.endswith(".origen.yaml") else
                  "train y val son la MISMA carpeta")
        print(f"  [!] {motivo}: las métricas saldrían infladas. Divido {int((1-val)*100)}/{int(val*100)}.")
        imgs = [i for c in train for i in _imagenes(c)]
        destino = _dividir(imgs, datos / f"{nombre}_dividido", clases, val=val)
        train, valid = [destino / "images" / "train"], [destino / "images" / "val"]
        base = destino

    # ---- Colab: copiar a disco local (leer de Drive es lentísimo) -------- #
    if E["tipo"] == "colab" and copiar_local and str(train[0]).startswith("/content/drive"):
        todas = train + valid + [_carpeta_labels(c) for c in train + valid]
        comun = Path(os.path.commonpath([str(p) for p in todas]))
        local = Path("/content/datasets") / nombre
        if not local.exists():
            print("  Copiando el dataset de Drive al disco de Colab (entrena mucho más rápido)...")
            shutil.copytree(comun, local)
        train = [local / p.relative_to(comun) for p in train]
        valid = [local / p.relative_to(comun) for p in valid]

    # ---- yaml normalizado: rutas absolutas, sin ambigüedad -------------- #
    final = {"train": [str(p) for p in train] if len(train) > 1 else str(train[0]),
             "val": [str(p) for p in valid] if len(valid) > 1 else str(valid[0]),
             "names": dict(enumerate(clases))}
    (datos / "_yaml").mkdir(exist_ok=True)
    salida = datos / "_yaml" / f"{nombre}.yaml"
    salida.write_text(yaml.safe_dump(final, allow_unicode=True, sort_keys=False), encoding="utf-8")
    return salida


def revisar_dataset(ruta_yaml: Path) -> dict:
    """Cuenta imágenes, etiquetas e instancias, y avisa lo que suele salir mal."""
    cfg = yaml.safe_load(Path(ruta_yaml).read_text(encoding="utf-8"))
    clases = _nombres_clases(cfg["names"])
    R = {"yaml": Path(ruta_yaml), "clases": clases, "splits": {}, "instancias": {},
         "formatos": set(), "problemas": [], "avisos": []}
    for split in ("train", "val"):
        carpetas = cfg[split] if isinstance(cfg[split], list) else [cfg[split]]
        n_img = n_sin = n_vacias = 0
        for c in map(Path, carpetas):
            for img in _imagenes(c):
                n_img += 1
                lbl = _carpeta_labels(c) / f"{img.stem}.txt"
                if not lbl.exists():
                    n_sin += 1
                    continue
                lineas = [l.split() for l in lbl.read_text(encoding="utf-8").splitlines() if l.strip()]
                if not lineas:
                    n_vacias += 1
                for campos in lineas:
                    try:
                        cid = int(float(campos[0]))
                    except (ValueError, IndexError):
                        R["problemas"].append(f"Línea ilegible en {lbl.name}: {' '.join(campos)[:40]}")
                        continue
                    R["instancias"][cid] = R["instancias"].get(cid, 0) + 1
                    R["formatos"].add("caja" if len(campos) == 5 else "polígono")
        R["splits"][split] = {"imagenes": n_img, "sin_etiqueta": n_sin, "vacias": n_vacias}
        if n_img == 0:
            R["problemas"].append(f"No hay imágenes en '{split}'.")
        if n_sin:
            R["avisos"].append(f"{split}: {n_sin} imágenes sin .txt. YOLO las toma como "
                               "fondo (sin objetos). Si no era la intención, etiquétalas.")

    fuera = sorted(i for i in R["instancias"] if i >= len(clases) or i < 0)
    if fuera:
        R["problemas"].append(f"Hay etiquetas con clase {fuera}, pero el yaml solo declara "
                              f"{len(clases)} clases (0 a {len(clases)-1}).")
    vacias = [c for i, c in enumerate(clases) if not R["instancias"].get(i)]
    if vacias:
        R["avisos"].append(f"Clases sin ningún ejemplo: {vacias}")
    if len(R["instancias"]) > 1:
        mx, mn = max(R["instancias"].values()), min(R["instancias"].values())
        if mx / max(mn, 1) > 10:
            R["avisos"].append(f"Desbalanceado: la clase más común tiene {mx // max(mn, 1)}× "
                               "más ejemplos que la más rara. Espera bajo recall en la rara.")
    n_tr = R["splits"]["train"]["imagenes"]
    if 0 < n_tr < 50:
        R["avisos"].append(f"Solo {n_tr} imágenes de entrenamiento: usa 'n' o 's' y no esperes milagros.")
    if "polígono" in R["formatos"]:
        R["avisos"].append("Las etiquetas son polígonos (como las de SAM 3). Sirven para "
                           "'segment' y también para 'detect': YOLO saca la caja del polígono.")

    print(f"Dataset: {R['yaml']}")
    print(f"  Clases ({len(clases)}): {', '.join(clases)}")
    for s, d in R["splits"].items():
        print(f"  {s:<5}: {d['imagenes']:>5} imágenes  ({d['sin_etiqueta']} sin etiqueta, {d['vacias']} vacías)")
    print("  Instancias: " + ", ".join(
        f"{clases[i] if i < len(clases) else '¿' + str(i) + '?'}={n}" for i, n in sorted(R["instancias"].items())))
    for p in R["problemas"]:
        print(f"  [X] {p}")
    for a in R["avisos"]:
        print(f"  [!] {a}")
    if not R["problemas"]:
        print("  Listo para entrenar.")
    return R


def ver_muestras(ruta_yaml: Path, n: int = 8, split: str = "train", semilla: int = 0):
    """Dibuja las etiquetas sobre algunas imágenes: la mejor forma de ver si están bien."""
    import cv2
    import numpy as np
    import matplotlib.pyplot as plt
    cfg = yaml.safe_load(Path(ruta_yaml).read_text(encoding="utf-8"))
    clases = _nombres_clases(cfg["names"])
    carpetas = cfg[split] if isinstance(cfg[split], list) else [cfg[split]]
    imgs = [i for c in map(Path, carpetas) for i in _imagenes(c)]
    random.Random(semilla).shuffle(imgs)
    imgs = imgs[:n]
    if not imgs:
        print("No hay imágenes que mostrar.")
        return
    paleta = plt.cm.tab10.colors
    cols = min(4, len(imgs))
    filas = (len(imgs) + cols - 1) // cols
    fig, axs = plt.subplots(filas, cols, figsize=(4 * cols, 4 * filas), squeeze=False)
    for ax in axs.flat:
        ax.axis("off")
    for ax, ruta in zip(axs.flat, imgs):
        img = cv2.cvtColor(cv2.imread(str(ruta)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        lbl = _carpeta_labels(ruta.parent) / f"{ruta.stem}.txt"
        if lbl.exists():
            for ln in lbl.read_text(encoding="utf-8").splitlines():
                v = ln.split()
                if len(v) < 5:
                    continue
                cid = int(float(v[0]))
                col = tuple(int(255 * x) for x in paleta[cid % 10])
                nums = np.array(v[1:], float)
                if len(v) == 5:
                    cx, cy, bw, bh = nums * [w, h, w, h]
                    p0, p1 = (int(cx - bw / 2), int(cy - bh / 2)), (int(cx + bw / 2), int(cy + bh / 2))
                else:
                    pts = (nums.reshape(-1, 2) * [w, h]).astype(np.int32)
                    cv2.polylines(img, [pts], True, col, max(1, w // 300))
                    p0, p1 = tuple(pts.min(0)), tuple(pts.max(0))
                cv2.rectangle(img, p0, p1, col, max(1, w // 250))
                nombre = clases[cid] if cid < len(clases) else f"?{cid}"
                cv2.putText(img, nombre, (p0[0], max(12, p0[1] - 4)), cv2.FONT_HERSHEY_SIMPLEX,
                            max(0.4, w / 900), col, max(1, w // 400))
        ax.imshow(img)
        ax.set_title(ruta.name, fontsize=8)
    plt.tight_layout()
    plt.show()


# --------------------------------------------------------------------------- #
# Turnos en la máquina compartida
# --------------------------------------------------------------------------- #
@contextmanager
def turno_gpu(E: dict, quien: str):
    """En la DGX solo entrena una persona a la vez; los demás esperan su turno."""
    if not E.get("turnos"):
        yield
        return
    try:
        import fcntl
    except ImportError:          # Windows: no hay fcntl, no hay turnos
        yield
        return
    ruta = Path(os.environ.get("VA_CANDADO", "/tmp/va_yolo_gpu.lock"))
    try:
        f = open(ruta, "a+")
        try:
            os.chmod(ruta, 0o666)        # que otros usuarios de la máquina también puedan usarlo
        except OSError:
            pass
    except OSError as e:
        print(f"(No pude usar el candado de turnos {ruta}: {e}. Entreno sin turno.)")
        yield
        return
    try:
        try:
            fcntl.flock(f, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError:
            f.seek(0)
            print(f"Esperando turno: está entrenando {f.read().strip() or 'alguien más'}...")
            t0 = time.time()
            fcntl.flock(f, fcntl.LOCK_EX)
            print(f"Tu turno (esperaste {(time.time() - t0) / 60:.1f} min).")
        f.seek(0)
        f.truncate()
        f.write(f"{quien} desde {time.strftime('%H:%M')}")
        f.flush()
        yield
    finally:
        try:
            f.seek(0)
            f.truncate()
            fcntl.flock(f, fcntl.LOCK_UN)
        finally:
            f.close()


# --------------------------------------------------------------------------- #
# Entrenamiento
# --------------------------------------------------------------------------- #
_ALIAS = {"metrics/mAP50-95(B)": "mAP50-95", "metrics/mAP50(B)": "mAP50",
          "metrics/precision(B)": "precision", "metrics/recall(B)": "recall",
          "metrics/mAP50-95(M)": "mAP50-95_mask", "metrics/mAP50(M)": "mAP50_mask"}


def _grafica_en_vivo():
    """Devuelve una función que redibuja las curvas al final de cada época."""
    import matplotlib.pyplot as plt
    from IPython.display import display
    hist = []
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
    plt.close(fig)
    h = display(fig, display_id=True)

    from matplotlib.ticker import MaxNLocator

    def actualizar(fila):
        hist.append(fila)
        ep = [r["epoch"] for r in hist]
        a1.clear(); a2.clear()
        for k, est in (("train/box_loss", "-"), ("val/box_loss", "--")):
            if any(k in r for r in hist):
                a1.plot(ep, [r.get(k, float("nan")) for r in hist], est, marker=".", label=k)
        a1.set_title("Pérdida de caja (baja = mejor)"); a1.set_xlabel("época"); a1.legend(fontsize=8)
        for k in ("mAP50", "mAP50-95"):
            if k in fila:
                a2.plot(ep, [r.get(k) for r in hist], marker=".", label=k)
        a2.set_title("mAP en validación (sube = mejor)"); a2.set_xlabel("época"); a2.legend(fontsize=8)
        a2.set_ylim(0, 1)
        for a in (a1, a2):
            a.xaxis.set_major_locator(MaxNLocator(integer=True))
        fig.tight_layout()
        h.update(fig)
    return actualizar


def _callbacks(modelo, meta_path: Path, en_vivo: bool):
    graf = _grafica_en_vivo() if en_vivo else None

    def al_terminar_epoca(trainer):
        if trainer.epoch >= trainer.epochs:      # la evaluación final repite la última época
            return
        try:
            fila = {"epoch": int(trainer.epoch) + 1}
            fila.update({_ALIAS.get(k.strip(), k.strip()): float(v) for k, v in (trainer.metrics or {}).items()})
            fila.update({k.strip(): float(v) for k, v in (trainer.label_loss_items(trainer.tloss) or {}).items()})
            if graf:
                graf(fila)
            meta = json.loads(meta_path.read_text(encoding="utf-8"))
            meta["epoca_actual"] = fila["epoch"]
            meta_path.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
        except Exception as e:           # una gráfica nunca debe tumbar el entrenamiento
            print(f"(no pude actualizar la gráfica: {e})")
    modelo.add_callback("on_fit_epoch_end", al_terminar_epoca)


def _guardar_meta(ruta: Path, **cambios):
    meta = json.loads(ruta.read_text(encoding="utf-8")) if ruta.exists() else {}
    meta.update(cambios)
    ruta.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
    return meta


def entrenar(E: dict, data: Path, version: str, size: str, tarea: str = "detect",
             epocas: int = 50, imgsz: int = 640, batch: int = 0, alumno: str = "",
             paciencia: int = 20, semilla: int = 0, en_vivo: bool = True, **extra) -> Path:
    """Entrena y deja todo lo necesario para comparar y para reanudar."""
    from ultralytics import YOLO

    pesos = nombre_pesos(version, size, tarea)
    cfg = yaml.safe_load(Path(data).read_text(encoding="utf-8"))
    n_train = sum(len(_imagenes(Path(c))) for c in (cfg["train"] if isinstance(cfg["train"], list) else [cfg["train"]]))
    if not batch or batch <= 0:
        batch = batch_automatico(E, size, imgsz, n_train)
    alumno = _limpiar(alumno) if alumno else ""
    nombre = f"{alumno + '_' if alumno else ''}{version}{size}-{tarea}_e{epocas}_{time.strftime('%m%d-%H%M%S')}"
    carpeta = E["runs"] / nombre
    carpeta.mkdir(parents=True, exist_ok=True)
    meta_path = carpeta / "meta_clase.json"
    _guardar_meta(meta_path, nombre=nombre, alumno=alumno, version=version, size=size,
                  tarea=tarea, pesos_iniciales=pesos, dataset=Path(data).stem, data=str(data),
                  epocas=epocas, imgsz=imgsz, batch=batch, equipo=E["tipo"],
                  dispositivo=E["dispositivo"], params_M=PARAMS_M.get(version, {}).get(size),
                  fecha=time.strftime("%Y-%m-%d %H:%M"), estado="en curso", minutos=0.0)

    # Guardar checkpoints seguido donde la sesión se cae (Colab)
    if E["tipo"] == "colab":
        extra.setdefault("save_period", 5)

    print(f"Modelo {pesos} | {epocas} épocas | imgsz {imgsz} | batch {batch} | "
          f"dispositivo {E['dispositivo']}")
    print(f"Corrida: {nombre}\n")

    modelo = YOLO(str(E["pesos"] / pesos))
    _callbacks(modelo, meta_path, en_vivo)
    t0 = time.time()
    try:
        with turno_gpu(E, alumno or nombre):
            t0 = time.time()
            args = dict(data=str(data), epochs=epocas, imgsz=imgsz, batch=batch,
                        device=E["dispositivo"], workers=E["workers"],
                        project=str(E["runs"]), name=nombre, exist_ok=True,
                        patience=paciencia, seed=semilla, plots=True)
            args.update(extra)          # lo que pases de más gana (workers=0, lr0=..., etc.)
            modelo.train(**args)
        estado = "terminado"
    except KeyboardInterrupt:
        estado = "interrumpido"
        print("\nInterrumpido. Puedes retomarlo en el Paso 7.")
    minutos = round((time.time() - t0) / 60, 2)
    _guardar_meta(meta_path, estado=estado, minutos=minutos)
    print(f"\n{estado.capitalize()} en {minutos:.1f} min  ->  {carpeta}")
    return carpeta


def corridas(E: dict) -> list[Path]:
    """Carpetas de corridas (las que tienen metadatos o resultados), de la más vieja a la más nueva."""
    return sorted((p for p in E["runs"].iterdir()
                   if p.is_dir() and ((p / "meta_clase.json").exists() or (p / "results.csv").exists())),
                  key=lambda p: p.stat().st_mtime)


def elegir_corrida(E: dict, nombre: str | None = None) -> Path:
    """La corrida indicada o, si no dices cuál, la más reciente."""
    if nombre:
        c = Path(nombre) if Path(nombre).is_absolute() else E["runs"] / nombre
        if not c.exists():
            raise FileNotFoundError(f"No existe la corrida {c}")
        return c
    todas = [c for c in corridas(E) if (c / "results.csv").exists()]
    if not todas:
        raise FileNotFoundError("Todavía no hay corridas con resultados. Entrena primero (Paso 6).")
    return todas[-1]


def reanudables(E: dict) -> list[Path]:
    out = []
    for c in corridas(E):
        meta = c / "meta_clase.json"
        estado = json.loads(meta.read_text(encoding="utf-8")).get("estado") if meta.exists() else "?"
        if (c / "weights" / "last.pt").exists() and estado != "terminado":
            out.append(c)
    return out


def reanudar(E: dict, carpeta: Path | str | None = None, en_vivo: bool = True) -> Path:
    """Retoma una corrida que se cortó, desde su último checkpoint."""
    from ultralytics import YOLO
    if carpeta is None:
        pend = reanudables(E)
        if not pend:
            print("No hay corridas pendientes de reanudar.")
            return None
        carpeta = pend[-1]
    carpeta = Path(carpeta)
    if not carpeta.is_absolute():
        carpeta = E["runs"] / carpeta
    last = carpeta / "weights" / "last.pt"
    if not last.exists():
        raise FileNotFoundError(f"No hay checkpoint en {last}")
    meta_path = carpeta / "meta_clase.json"
    previo = json.loads(meta_path.read_text(encoding="utf-8")).get("minutos", 0) if meta_path.exists() else 0
    print(f"Reanudando {carpeta.name} desde {last.name}\n")
    modelo = YOLO(str(last))
    if meta_path.exists():
        _callbacks(modelo, meta_path, en_vivo)
    t0 = time.time()
    try:
        with turno_gpu(E, carpeta.name):
            t0 = time.time()
            modelo.train(resume=True, device=E["dispositivo"], workers=E["workers"])
        estado = "terminado"
    except KeyboardInterrupt:
        estado = "interrumpido"
    # El tiempo se acumula para que la comparación siga siendo honesta
    _guardar_meta(meta_path, estado=estado, minutos=round(previo + (time.time() - t0) / 60, 2))
    print(f"\n{estado.capitalize()}  ->  {carpeta}")
    return carpeta


# --------------------------------------------------------------------------- #
# Resultados y comparación
# --------------------------------------------------------------------------- #
def cargar_curvas(carpeta: Path):
    import pandas as pd
    csv = Path(carpeta) / "results.csv"
    if not csv.exists():
        return None
    df = pd.read_csv(csv)
    df.columns = [c.strip() for c in df.columns]
    return df.rename(columns={k: v for k, v in _ALIAS.items() if k in df.columns})


def ver_curvas(carpeta: Path):
    import matplotlib.pyplot as plt
    df = cargar_curvas(carpeta)
    if df is None:
        print("Esa corrida no tiene results.csv todavía.")
        return
    perdidas = [c.split("/")[1] for c in df.columns if c.startswith("train/")]
    fig, axs = plt.subplots(1, len(perdidas) + 1, figsize=(4.2 * (len(perdidas) + 1), 3.4))
    for ax, p in zip(axs, perdidas):
        ax.plot(df["epoch"], df[f"train/{p}"], label="train")
        if f"val/{p}" in df:
            ax.plot(df["epoch"], df[f"val/{p}"], "--", label="val")
        ax.set_title(p); ax.set_xlabel("época"); ax.legend(fontsize=8)
    ax = axs[-1]
    for m in ("precision", "recall", "mAP50", "mAP50-95", "mAP50-95_mask"):
        if m in df:
            ax.plot(df["epoch"], df[m], label=m)
    ax.set_ylim(0, 1); ax.set_title("métricas en val"); ax.set_xlabel("época"); ax.legend(fontsize=8)
    from matplotlib.ticker import MaxNLocator
    for a in axs:
        a.xaxis.set_major_locator(MaxNLocator(integer=True))
    fig.suptitle(Path(carpeta).name, fontsize=10)
    plt.tight_layout(); plt.show()
    mejor = df.loc[df["mAP50-95"].idxmax()] if "mAP50-95" in df else None
    if mejor is not None:
        print(f"Mejor época: {int(mejor['epoch'])}  ·  mAP50 {mejor['mAP50']:.3f}  ·  "
              f"mAP50-95 {mejor['mAP50-95']:.3f}  ·  precision {mejor['precision']:.3f}  ·  "
              f"recall {mejor['recall']:.3f}")


def ver_imagenes(carpeta: Path, *nombres: str):
    """Muestra las imágenes que Ultralytics guarda en la carpeta de la corrida."""
    import matplotlib.pyplot as plt
    import matplotlib.image as mpimg
    rutas = [Path(carpeta) / n for n in nombres if (Path(carpeta) / n).exists()]
    if not rutas:
        print("No encontré esas imágenes en la corrida.")
        return
    fig, axs = plt.subplots(1, len(rutas), figsize=(7 * len(rutas), 6), squeeze=False)
    for ax, r in zip(axs[0], rutas):
        ax.imshow(mpimg.imread(r)); ax.set_title(r.name, fontsize=9); ax.axis("off")
    plt.tight_layout(); plt.show()


def probar(carpeta: Path, fuente, conf: float = 0.25, n: int = 8, dispositivo: str | None = None):
    """Corre el mejor modelo sobre imágenes nuevas y las muestra."""
    import matplotlib.pyplot as plt
    from ultralytics import YOLO
    best = Path(carpeta) / "weights" / "best.pt"
    modelo = YOLO(str(best))
    fuente = Path(fuente)
    imgs = _imagenes(fuente)[:n] if fuente.is_dir() else [fuente]
    if not imgs:
        print(f"No hay imágenes en {fuente}")
        return
    res = modelo.predict([str(i) for i in imgs], conf=conf, verbose=False,
                         **({"device": dispositivo} if dispositivo else {}))
    cols = min(4, len(res))
    filas = (len(res) + cols - 1) // cols
    fig, axs = plt.subplots(filas, cols, figsize=(4 * cols, 4 * filas), squeeze=False)
    for ax in axs.flat:
        ax.axis("off")
    for ax, r in zip(axs.flat, res):
        ax.imshow(r.plot()[:, :, ::-1])
        ax.set_title(f"{Path(r.path).name}  ({len(r.boxes)} obj.)", fontsize=8)
    plt.tight_layout(); plt.show()


def tabla_comparativa(E: dict, dataset: str | None = None):
    """Una fila por corrida. Si das 'dataset', solo compara las de ese dataset."""
    import pandas as pd
    filas = []
    for c in corridas(E):
        df = cargar_curvas(c)
        if df is None or df.empty:
            continue
        meta_p = c / "meta_clase.json"
        m = json.loads(meta_p.read_text(encoding="utf-8")) if meta_p.exists() else {}
        if dataset and m.get("dataset") not in (None, dataset):
            continue
        f = {"corrida": c.name, "alumno": m.get("alumno", ""),
             "modelo": f"{m.get('version', '?')}{m.get('size', '')}", "tarea": m.get("tarea", ""),
             "dataset": m.get("dataset", ""), "equipo": m.get("equipo", ""),
             "épocas": int(df["epoch"].max()), "imgsz": m.get("imgsz"), "batch": m.get("batch"),
             "params_M": m.get("params_M"), "minutos": m.get("minutos"), "estado": m.get("estado", "")}
        # Todas las métricas de la MISMA época: la del mejor mAP50-95
        mejor = df.loc[df["mAP50-95"].idxmax()] if "mAP50-95" in df else df.iloc[-1]
        f["mejor_época"] = int(mejor["epoch"])
        for k in ("mAP50-95", "mAP50", "precision", "recall", "mAP50-95_mask"):
            if k in df:
                f[k] = round(float(mejor[k]), 3)
        filas.append(f)
    if not filas:
        print("Todavía no hay corridas con resultados.")
        return pd.DataFrame()
    return pd.DataFrame(filas).sort_values("mAP50-95", ascending=False).reset_index(drop=True)


def graficar_comparacion(E: dict, tabla, top: int = 8):
    import matplotlib.pyplot as plt
    if tabla is None or tabla.empty:
        return
    t = tabla.head(top)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.2))
    for _, f in t.iterrows():
        df = cargar_curvas(E["runs"] / f["corrida"])
        etiqueta = f"{f['alumno'] + ' · ' if f['alumno'] else ''}{f['modelo']}"
        a1.plot(df["epoch"], df["mAP50-95"], label=etiqueta)
        if f["minutos"]:
            a2.scatter(f["minutos"], f["mAP50-95"], s=30 + 6 * (f["params_M"] or 3))
            a2.annotate(etiqueta, (f["minutos"], f["mAP50-95"]), fontsize=8,
                        xytext=(4, 4), textcoords="offset points")
    a1.set_title("mAP50-95 por época"); a1.set_xlabel("época"); a1.legend(fontsize=7)
    from matplotlib.ticker import MaxNLocator
    a1.xaxis.set_major_locator(MaxNLocator(integer=True))
    a2.set_title("¿Vale la pena el tiempo extra?  (tamaño = parámetros)")
    a2.set_xlabel("minutos de entrenamiento"); a2.set_ylabel("mejor mAP50-95")
    plt.tight_layout(); plt.show()


print("Funciones de apoyo listas.")

---
## Paso 3 — ¿Dónde estoy corriendo?

Lee los avisos `[!]`: ahí sale si te falta prender la GPU en Colab o si estás en la
máquina compartida. En Colab te va a pedir permiso para conectar tu Drive, que es
donde se guardan tus corridas para que no se pierdan al desconectarse.

In [ ]:
E = preparar_entorno(DISPOSITIVO)

---
## Paso 4 — Tu dataset

`DATASET` acepta lo que probablemente ya tienes:

| Tienes | Pon en `DATASET` |
|---|---|
| Nada todavía | `"demo"` |
| Export de **Roboflow** (formato YOLOv8/YOLO11) | la ruta del `.zip` |
| Salida del script de **SAM 3** | la carpeta `yolo/` que generó, o su `data.yaml` |
| Tu propio `data.yaml` | su ruta |
| Carpeta con `images/` y `labels/` sin yaml | la carpeta, y llena `CLASES` |

**Dónde poner los archivos:**
- **Colab:** súbelos a tu Drive en `VisionArtificial/entrenar_yolo/datos/` y escribe solo el
  nombre (`"humo.zip"`). El notebook los copia al disco de Colab antes de entrenar, porque
  leer miles de imágenes directo de Drive es lentísimo.
- **Tu computadora o la Spark:** en la carpeta `datos/`, junto a este notebook.

**Lo que se arregla solo:** si `train` y `val` son la misma carpeta (así sale el export de
SAM 3) o no hay `val`, se divide 80/20. Con `train` = `val` el modelo se calificaría con
imágenes que ya vio y el mAP saldría inflado. Las rutas relativas raras (las de Roboflow
con `../`) también se corrigen.

**Mira las muestras de abajo.** Si las cajas no caen sobre los objetos, el problema está
en las etiquetas y ningún modelo lo va a arreglar.

In [ ]:
DATA = preparar_dataset(E, DATASET, CLASES)
REPORTE = revisar_dataset(DATA)
ver_muestras(DATA, n=8)

---
## Paso 5 — Elegir modelo

Las versiones no son "mejores" solo por ser más nuevas: cambian la arquitectura, la
velocidad y lo que traen preentrenado. Los tamaños van de `n` (nano, rápido) a `x`
(enorme, lento). La pregunta de ingeniería no es cuál es el mejor, sino **cuál es
suficiente** para tu problema y tu hardware.

In [ ]:
display(mostrar_catalogo())

PESOS = nombre_pesos(VERSION, TAMANO, TAREA)     # truena aquí si la combinación no existe
n_train = REPORTE["splits"]["train"]["imagenes"]
b = BATCH if BATCH > 0 else batch_automatico(E, TAMANO, IMGSZ, n_train)
print(f"\nVas a entrenar {PESOS}  ·  {PARAMS_M[VERSION][TAMANO]} M de parámetros  ·  "
      f"batch {b}{' (automático)' if BATCH <= 0 else ''}  ·  {EPOCAS} épocas  ·  imgsz {IMGSZ}")

---
## Paso 6 — Entrenar

Mientras entrena verás dos gráficas que se actualizan cada época:

- **Pérdida de caja** (`box_loss`): qué tan mal ubica las cajas. Debe bajar. Si la de
  `train` baja pero la de `val` empieza a subir, el modelo está **memorizando** (sobreajuste).
- **mAP50 y mAP50-95**: qué tan bien detecta en validación, de 0 a 1. `mAP50` acepta cajas
  que se traslapan al 50 % con la real; `mAP50-95` es más estricto y es el que usamos para comparar.

Si no mejora en 20 épocas seguidas se detiene solo (`patience`). Para detenerlo tú,
usa el botón de *stop*: lo que lleva queda guardado.

En la **DGX Spark** solo entrena una persona a la vez; si alguien está usando la GPU,
la celda dice *Esperando turno* y arranca sola cuando se libera.

In [ ]:
assert not REPORTE["problemas"], "Arregla los problemas [X] del dataset (Paso 4) antes de entrenar."

RUN = entrenar(E, DATA, VERSION, TAMANO, TAREA,
               epocas=EPOCAS, imgsz=IMGSZ, batch=BATCH, alumno=ALUMNO)

# Más parámetros de Ultralytics van al final, por ejemplo:
#   entrenar(..., lr0=0.005, fliplr=0.0, mosaic=0.5, cache="ram")

---
## Paso 7 — Si se cortó (sobre todo en Colab)

Colab se desconecta por inactividad o por tiempo, y es normal. Como se guarda un
checkpoint cada 5 épocas en tu Drive, no empiezas de cero:

1. Vuelve a conectar y corre los **Pasos 0 a 4** (el 4 vuelve a copiar el dataset).
2. Corre esta celda. Sin argumentos retoma la corrida más reciente que quedó a medias.

El tiempo se acumula, así que la comparación de minutos sigue siendo justa.

In [ ]:
pendientes = reanudables(E)
print("Corridas a medias:", [c.name for c in pendientes] or "ninguna")

# RUN = reanudar(E)                          # la más reciente
# RUN = reanudar(E, "nombre_de_la_corrida")  # una en particular

---
## Paso 8 — Resultados de la corrida

Si quieres ver una corrida anterior, pon su nombre: `RUN = elegir_corrida(E, "nombre")`.

Cómo leer cada cosa:
- **Curvas:** busca el punto donde `val` deja de mejorar. Entrenar más allá solo gasta tiempo.
- **Matriz de confusión:** la diagonal son aciertos. La fila/columna `background` son objetos
  que no vio (falsos negativos) o que inventó donde no había nada (falsos positivos).
- **Etiquetas contra predicciones:** el mismo lote de validación, a la izquierda lo que dice
  el dataset y a la derecha lo que dijo el modelo.

In [ ]:
RUN = RUN if "RUN" in globals() and RUN else elegir_corrida(E)
ver_curvas(RUN)
ver_imagenes(RUN, "confusion_matrix_normalized.png")
ver_imagenes(RUN, "val_batch0_labels.jpg", "val_batch0_pred.jpg")

**Métricas por clase.** El promedio esconde cosas: a veces una clase va perfecta y otra
casi no se detecta. Esta celda vuelve a evaluar el mejor modelo y separa por clase.

In [ ]:
import pandas as pd
from ultralytics import YOLO

met = YOLO(str(RUN / "weights" / "best.pt")).val(
    data=str(DATA), device=E["dispositivo"], workers=E["workers"],
    project=str(RUN), name="val_por_clase", exist_ok=True, plots=False, verbose=False)
display(pd.DataFrame(met.summary()).round(3))

---
## Paso 9 — Probarlo con imágenes nuevas

Por defecto usa imágenes de validación. Para usar las tuyas, pon la ruta de una carpeta
o de una imagen en `FUENTE`. Sube `CONF` si ves detecciones falsas; bájalo si se le escapan objetos.

En tu computadora también puedes probarlo con la **webcam** (no funciona en Colab):

```python
YOLO(str(RUN / "weights" / "best.pt")).predict(source=0, show=True)   # q para salir
```

In [ ]:
import yaml
FUENTE = yaml.safe_load(DATA.read_text())["val"]   # o "datos/mis_fotos/"
CONF = 0.25

probar(RUN, FUENTE[0] if isinstance(FUENTE, list) else FUENTE, conf=CONF, n=8,
       dispositivo=E["dispositivo"])

---
## Paso 10 — Comparar corridas

Aquí está el punto de todo el ejercicio. Cada vez que entrenas se agrega una fila.
Por defecto solo compara corridas del **mismo dataset**: comparar mAP entre datasets
distintos no significa nada.

- En la **DGX Spark** la tabla incluye las corridas de **todo el grupo**.
- En **Colab** y en tu computadora incluye las tuyas. Para comparar con tu equipo en Colab,
  compartan la carpeta `VisionArtificial` de Drive (ver README).

La gráfica de la derecha responde la pregunta que importa: **¿cuánto mAP gano por cada
minuto extra?** Si un modelo `m` tarda el triple que un `n` y gana 0.01, no vale la pena.

In [ ]:
TABLA = tabla_comparativa(E, dataset=DATA.stem)   # dataset=None para ver todas
display(TABLA)
graficar_comparacion(E, TABLA)

---
## Paso 11 — Llevarte el modelo (opcional)

`best.pt` es tu modelo entrenado; con eso basta para usarlo en Python. Para otros
programas o para una Jetson, expórtalo a ONNX (en la Jetson, de ahí se pasa a TensorRT).

En **Colab sin Drive**, descarga la carpeta de la corrida antes de cerrar o se pierde.

In [ ]:
best = RUN / "weights" / "best.pt"
print("Tu modelo:", best)

# onnx = YOLO(str(best)).export(format="onnx", imgsz=IMGSZ)
# print("Exportado:", onnx)

# Solo en Colab, para bajar toda la corrida en un zip:
# if EN_COLAB:
#     from google.colab import files
#     files.download(shutil.make_archive(f"/content/{RUN.name}", "zip", RUN))

---
## Ejercicios

Usa **el mismo dataset** en todos, si no, la comparación no vale.

1. **Tamaño.** Entrena `n`, `s` y `m` de la misma versión. ¿Cuánto mAP50-95 ganas en cada
   salto y cuánto tiempo te cuesta? Usa la gráfica del Paso 10.
2. **Versión.** Mismo tamaño, `yolov8` contra `yolo11` contra `yolo26`. ¿Lo nuevo gana en
   *tu* dataset o solo en COCO?
3. **Resolución.** Repite tu mejor modelo con `IMGSZ` 320 y 1024. ¿Qué clases cambian?
   (Pista: piensa en el tamaño de los objetos en píxeles.)
4. **Sobreajuste.** Entrena con 100 épocas y `patience` grande:
   `entrenar(..., paciencia=1000)`. Encuentra en las curvas la época donde `val` deja de
   mejorar mientras `train` sigue bajando.
5. **Decide como ingeniero.** Tienes que correr el detector en una Jetson a 30 cuadros por
   segundo. Con tu tabla, elige un modelo y justifica con mAP, tiempo y parámetros, no solo
   con el número más alto.